In [204]:
!pip3 install kagglehub
!pip3 install scikit-learn

In [205]:
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Download latest version
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Path to dataset files:", path)

Path to dataset files: /Users/pponali/.cache/kagglehub/datasets/manishkr1754/cardekho-used-car-data/versions/2


In [206]:
df = pd.read_csv(path + "/cardekho_dataset.csv")
df

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


## ** Task 1 **

In [207]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  str    
 2   brand              15411 non-null  str    
 3   model              15411 non-null  str    
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  str    
 7   fuel_type          15411 non-null  str    
 8   transmission_type  15411 non-null  str    
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), str(6)
memory usage: 1.6 MB


In [208]:
df.isnull().sum() * 100 / len(df)

Unnamed: 0           0.0
car_name             0.0
brand                0.0
model                0.0
vehicle_age          0.0
km_driven            0.0
seller_type          0.0
fuel_type            0.0
transmission_type    0.0
mileage              0.0
engine               0.0
max_power            0.0
seats                0.0
selling_price        0.0
dtype: float64

In [209]:
df.dropna(subset=['selling_price'], inplace=True)

In [210]:
df['mileage'] = df['mileage'].astype('str').str.replace(' kmpl', '').str.strip()
df['mileage'] = pd.to_numeric(df['mileage'], errors='coerce')


In [211]:

df['engine'] = (df['engine'].astype('str').str.replace('CC', '').str.strip())
df['engine'] =pd.to_numeric(df['engine'], errors='coerce')


In [212]:
df['max_power'] = df['max_power'].astype('str').str.replace(' bhp', '').str.strip()
df['max_power'] = pd.to_numeric(df['max_power'], errors='coerce')

In [213]:
for col in ['mileage', 'engine', 'max_power']:
    median_value = df[col].median()
    df[col].fillna(median_value)
    print(f"Missing values in {col} after imputation: {median_value}")    

Missing values in mileage after imputation: 19.67
Missing values in engine after imputation: 1248.0
Missing values in max_power after imputation: 88.5


In [214]:
df = df[(df['selling_price'] != 999999999) | (df['selling_price'] > 10000)]
df

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [215]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [216]:
print(f"Number of rows and columns after cleaning: rows are {df.shape[0]}. columns are {df.shape[1]}")

Number of rows and columns after cleaning: rows are 15411. columns are 14


# Task 2: Encode Categorical Features

In [217]:
df['transmission_type'] = df['transmission_type'].map({
    'Manual': 0,
    'Automatic': 1
})
df

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,0,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,0,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,0,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,0,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,0,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,0,19.81,1086,68.05,5,250000
15407,19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,0,17.50,1373,91.10,7,925000
15408,19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,0,21.14,1498,103.52,5,425000
15409,19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,0,16.00,2179,140.00,7,1225000


In [218]:
df = pd.get_dummies(df, columns=['seller_type'], drop_first=True)

In [219]:
df = pd.get_dummies(df, columns=['fuel_type'], drop_first=True)


In [220]:
df

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,transmission_type,mileage,engine,max_power,seats,selling_price,seller_type_Individual,seller_type_Trustmark Dealer,fuel_type_Diesel,fuel_type_Electric,fuel_type_LPG,fuel_type_Petrol
0,0,Maruti Alto,Maruti,Alto,9,120000,0,19.70,796,46.30,5,120000,True,False,False,False,False,True
1,1,Hyundai Grand,Hyundai,Grand,5,20000,0,18.90,1197,82.00,5,550000,True,False,False,False,False,True
2,2,Hyundai i20,Hyundai,i20,11,60000,0,17.00,1197,80.00,5,215000,True,False,False,False,False,True
3,3,Maruti Alto,Maruti,Alto,9,37000,0,20.92,998,67.10,5,226000,True,False,False,False,False,True
4,4,Ford Ecosport,Ford,Ecosport,6,30000,0,22.77,1498,98.59,5,570000,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,19537,Hyundai i10,Hyundai,i10,9,10723,0,19.81,1086,68.05,5,250000,False,False,False,False,False,True
15407,19540,Maruti Ertiga,Maruti,Ertiga,2,18000,0,17.50,1373,91.10,7,925000,False,False,False,False,False,True
15408,19541,Skoda Rapid,Skoda,Rapid,6,67000,0,21.14,1498,103.52,5,425000,False,False,True,False,False,False
15409,19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,0,16.00,2179,140.00,7,1225000,False,False,True,False,False,False


In [221]:
print(df.columns)
for col in df.columns:
    print(f"Column name : {col} and the data type is {df[col].dtype}")

Index(['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven',
       'transmission_type', 'mileage', 'engine', 'max_power', 'seats',
       'selling_price', 'seller_type_Individual',
       'seller_type_Trustmark Dealer', 'fuel_type_Diesel',
       'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol'],
      dtype='str')
Column name : Unnamed: 0 and the data type is int64
Column name : car_name and the data type is str
Column name : brand and the data type is str
Column name : model and the data type is str
Column name : vehicle_age and the data type is int64
Column name : km_driven and the data type is int64
Column name : transmission_type and the data type is int64
Column name : mileage and the data type is float64
Column name : engine and the data type is int64
Column name : max_power and the data type is float64
Column name : seats and the data type is int64
Column name : selling_price and the data type is int64
Column name : seller_type_Individual and the dat

In [222]:
df.columns.to_list

<bound method IndexOpsMixin.tolist of Index(['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven',
       'transmission_type', 'mileage', 'engine', 'max_power', 'seats',
       'selling_price', 'seller_type_Individual',
       'seller_type_Trustmark Dealer', 'fuel_type_Diesel',
       'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol'],
      dtype='str')>

# Why is drop_first=True used in one-hot encoding, and what does a row of all zeros in the new dummy columns represent?

We use drop_first= True in one-hot encoding to reduce multicollinerity which confised the linear model.

# Task 3 — Split and Compute Baseline MAE

In [223]:
	

df = df.drop(columns=['Unnamed: 0','car_name','brand','model'])

x = df.drop(columns=['selling_price'])
y = df['selling_price']

my_model = LinearRegression()


x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.20, random_state=42)

#TRINING THE MODEL WITH FIT METHOD
my_model.fit(x_train, y_train)
# Predicting the values against the test data.
y_pred = my_model.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"Baseline MAE: ₹{mae:.2f}")




Baseline MAE: ₹278351.63
